<a href="https://www.kaggle.com/code/dyyahqurniatun/titanic-competition?scriptVersionId=339397478" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


# EDA

In [2]:
# load the dataset
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")

In [3]:
# look at the first five rows of the data
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# get the data summary
train_data.info()
train_data.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
# check the missing values
train_data.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [6]:
train_data['Age'].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

## **Handle Missing Values**

In [7]:
# Fill missing values

df_filled = train_data.copy()
df_filled['Age'] = df_filled['Age'].fillna(df_filled['Age'].median())
df_filled['Embarked'] = df_filled['Embarked'].fillna(df_filled['Embarked'].mode()[0])

## **Feature Engineering**

In [8]:
# Create a new DataFrame and perform feature engineering in one step
df_featured = df_filled.copy()

# Create AgeGroup
# Define age groups based on quartiles for balanced distribution.
bins = [0, 20, 28, 38, 80]
labels = ['0-20', '20-28', '28-38', '38-80']
df_featured['AgeGroup'] = pd.cut(df_featured['Age'], bins=bins, labels=labels, include_lowest=True)

# Create FamilySize
# Combine SibSp and Parch into a new column to represent family size.
df_featured['FamilySize'] = df_featured['SibSp'] + df_featured['Parch']

# Extract Title
# Extract the title from the Name column.
df_featured['Title'] = df_featured['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

In [9]:
df_featured.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,AgeGroup,FamilySize,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,20-28,1,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,28-38,1,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,20-28,0,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,28-38,1,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,28-38,0,Mr


## **Drop and Encode**

In [10]:
# Drop columns that we decided were unnecessary.
# PassengerId, Name, Ticket, Cabin, and the original Age column.

df_drop = df_featured.drop(['PassengerId', 'Name', 'Ticket', 'Cabin', 'Age'], axis=1)
df_drop.head()

,Survived,Pclass,Sex,SibSp,Parch,Fare,Embarked,AgeGroup,FamilySize,Title
0,0,3,male,1,0,7.2500,S,20-28,1,Mr
1,1,1,female,1,0,71.2833,C,28-38,1,Mrs
2,1,3,female,0,0,7.9250,S,20-28,0,Miss
3,1,1,female,1,0,53.1000,S,28-38,1,Mrs
4,0,3,male,0,0,8.0500,S,28-38,0,Mr


In [11]:
# Apply one-hot encoding to categorical columns

df_encoded = pd.get_dummies(df_drop, columns=['Sex', 'Embarked', 'AgeGroup', 'Title'], drop_first=True)
df_encoded.head()

,Survived,Pclass,SibSp,Parch,Fare,FamilySize,Sex_male,Embarked_Q,Embarked_S,AgeGroup_20-28,...,Title_Major,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir
0,0,3,1,0,7.2500,1,True,False,True,True,...,False,False,False,False,False,True,False,False,False,False
1,1,1,1,0,71.2833,1,False,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,1,3,0,0,7.9250,0,False,False,True,True,...,False,False,True,False,False,False,False,False,False,False
3,1,1,1,0,53.1000,1,False,False,True,False,...,False,False,False,False,False,False,True,False,False,False
4,0,3,0,0,8.0500,0,True,False,True,False,...,False,False,False,False,False,True,False,False,False,False


In [12]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 28 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Survived        891 non-null    int64  
 1   Pclass          891 non-null    int64  
 2   SibSp           891 non-null    int64  
 3   Parch           891 non-null    int64  
 4   Fare            891 non-null    float64
 5   FamilySize      891 non-null    int64  
 6   Sex_male        891 non-null    bool   
 7   Embarked_Q      891 non-null    bool   
 8   Embarked_S      891 non-null    bool   
 9   AgeGroup_20-28  891 non-null    bool   
 10  AgeGroup_28-38  891 non-null    bool   
 11  AgeGroup_38-80  891 non-null    bool   
 12  Title_Col       891 non-null    bool   
 13  Title_Countess  891 non-null    bool   
 14  Title_Don       891 non-null    bool   
 15  Title_Dr        891 non-null    bool   
 16  Title_Jonkheer  891 non-null    bool   
 17  Title_Lady      891 non-null    boo

# Split Data

In [13]:
# Split the Data into Training and Validation Sets

from sklearn.model_selection import train_test_split

# Separate features and target variable
X = df_encoded.drop('Survived', axis=1)  # Features
y = df_encoded['Survived']               # Target variable

# Split the data: 80% training and 20% validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


# Model Selection

In [14]:
# Import Model

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

## **Log Reg**

In [15]:
param_grid_logreg = {
    'C': [0.01, 0.1, 1, 10, 100],  # Regularization strength
    'solver': ['liblinear', 'saga'],  # Solver algorithms
    'penalty': ['l1', 'l2']  # Type of regularization
}

grid_search_logreg = GridSearchCV(LogisticRegression(max_iter=5000, random_state=42), param_grid_logreg, cv=5, scoring='accuracy')
grid_search_logreg.fit(X_train, y_train)

print("Best Logistic Regression parameters:", grid_search_logreg.best_params_)
print("Best cross-validated accuracy:", grid_search_logreg.best_score_)


Best Logistic Regression parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Best cross-validated accuracy: 0.8272431793558555


In [16]:
# Logistic Regression

logreg = LogisticRegression(C=10, 
                            penalty='l1', 
                            solver='liblinear', 
                            max_iter=5000, 
                            random_state=42)
logreg.fit(X_train, y_train)

LogisticRegression(C=10, max_iter=5000, penalty='l1', random_state=42,
                   solver='liblinear')

## **Random Forest**

In [17]:
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5, scoring='accuracy')
grid_search_rf.fit(X_train, y_train)

print("Best Random Forest parameters:", grid_search_rf.best_params_)
print("Best cross-validated accuracy:", grid_search_rf.best_score_)


Best Random Forest parameters: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}
Best cross-validated accuracy: 0.8370333891460652


In [18]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, 
                            max_depth=10, 
                            min_samples_split=10, 
                            random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, min_samples_split=10, random_state=42)

## **SVC**

In [19]:
# Support Vector Classifier
svc = SVC(kernel='linear', C=1, gamma=1, random_state=42)
svc.fit(X_train, y_train)

SVC(C=1, gamma=1, kernel='linear', random_state=42)

## **KNN**

In [20]:
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 10],
    'weights': ['uniform', 'distance'],  # Uniform: all points are weighted equally; Distance: closer points have higher weight
    'metric': ['euclidean', 'manhattan']
}

grid_search_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn, cv=5, scoring='accuracy')
grid_search_knn.fit(X_train, y_train)

print("Best KNN parameters:", grid_search_knn.best_params_)
print("Best cross-validated accuracy:", grid_search_knn.best_score_)

Best KNN parameters: {'metric': 'manhattan', 'n_neighbors': 5, 'weights': 'uniform'}
Best cross-validated accuracy: 0.7738796414852753


In [21]:
# K-Nearest Neighbors
knn = KNeighborsClassifier(n_neighbors=5, metric='manhattan', weights='uniform')
knn.fit(X_train, y_train)

KNeighborsClassifier(metric='manhattan')

## **XGBoost**

In [22]:
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search_xgb = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42), param_grid_xgb, cv=5, scoring='accuracy')
grid_search_xgb.fit(X_train, y_train)

print("Best XGBoost parameters:", grid_search_xgb.best_params_)
print("Best cross-validated accuracy:", grid_search_xgb.best_score_)


Best XGBoost parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}
Best cross-validated accuracy: 0.8398699891657639


In [23]:
# XGBoost
xgb = XGBClassifier(
    colsample_bytree= 0.8,
    n_estimators=50, 
    max_depth=3, 
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
xgb.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=50, n_jobs=None,
              num_parallel_tree=None, random_state=42, ...)

# Model Performance

In [24]:
# Evaluate Model Performance on Validation Set

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define the evaluation function
def evaluate_model(model_name, y_true, y_pred):
    print(f"{model_name} Performance:")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score: {f1_score(y_true, y_pred):.4f}\n")

# Logistic Regression Evaluation
logreg_pred = logreg.predict(X_val)
evaluate_model("Logistic Regression", y_val, logreg_pred)

# Random Forest Evaluation
rf_pred = rf.predict(X_val)
evaluate_model("Random Forest", y_val, rf_pred)

# Support Vector Classifier Evaluation
svc_pred = svc.predict(X_val)
evaluate_model("Support Vector Classifier", y_val, svc_pred)

# K-Nearest Neighbors Evaluation
knn_pred = knn.predict(X_val)
evaluate_model("K-Nearest Neighbors", y_val, knn_pred)

# XGBoost Evaluation
xgb_pred = xgb.predict(X_val)
evaluate_model("XGBoost", y_val, xgb_pred)


Logistic Regression Performance:
Accuracy: 0.8045
Precision: 0.7746
Recall: 0.7432
F1 Score: 0.7586

Random Forest Performance:
Accuracy: 0.8324
Precision: 0.8235
Recall: 0.7568
F1 Score: 0.7887

Support Vector Classifier Performance:
Accuracy: 0.8212
Precision: 0.7917
Recall: 0.7703
F1 Score: 0.7808

K-Nearest Neighbors Performance:
Accuracy: 0.8156
Precision: 0.7887
Recall: 0.7568
F1 Score: 0.7724

XGBoost Performance:
Accuracy: 0.8156
Precision: 0.7971
Recall: 0.7432
F1 Score: 0.7692



# On Test Data

## **Preprocess Test Data**

In [25]:
# Step 1: Handle Missing Values in `test_data`
test_data['Age'] = test_data['Age'].fillna(train_data['Age'].median())  # Use median from train_data
test_data['Fare'] = test_data['Fare'].fillna(train_data['Fare'].median())  # Use median from train_data
test_data['Embarked'] = test_data['Embarked'].fillna(train_data['Embarked'].mode()[0])  # Use mode from train_data

# Step 2: Feature Engineering (Add AgeGroup, FamilySize, and Title)
test_data['AgeGroup'] = pd.cut(test_data['Age'], bins=bins, labels=labels, include_lowest=True)
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch']
test_data['Title'] = test_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Step 3: Encode Categorical Variables
test_data_encoded = pd.get_dummies(test_data, columns=['Sex', 'Embarked', 'AgeGroup', 'Title'], drop_first=True)
test_data_encoded = test_data_encoded.reindex(columns=X_train.columns, fill_value=0)


In [26]:
test_data_encoded.head()
test_data_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Pclass          418 non-null    int64  
 1   SibSp           418 non-null    int64  
 2   Parch           418 non-null    int64  
 3   Fare            418 non-null    float64
 4   FamilySize      418 non-null    int64  
 5   Sex_male        418 non-null    bool   
 6   Embarked_Q      418 non-null    bool   
 7   Embarked_S      418 non-null    bool   
 8   AgeGroup_20-28  418 non-null    bool   
 9   AgeGroup_28-38  418 non-null    bool   
 10  AgeGroup_38-80  418 non-null    bool   
 11  Title_Col       418 non-null    int64  
 12  Title_Countess  418 non-null    int64  
 13  Title_Don       418 non-null    int64  
 14  Title_Dr        418 non-null    bool   
 15  Title_Jonkheer  418 non-null    int64  
 16  Title_Lady      418 non-null    int64  
 17  Title_Major     418 non-null    int

## **Prediction**

In [27]:
# Using the trained Random Forest model to predict survival on test_data_encoded
test_predictions = rf.predict(test_data_encoded)

# Show the first 10 predictions along with their PassengerId
predictions_df = pd.DataFrame({
    'PassengerId': test_data['PassengerId'],
    'Predicted Survived': test_predictions
})

# Display the first few predictions
print(predictions_df.head(10))  


   PassengerId  Predicted Survived
0          892                   0
1          893                   0
2          894                   0
3          895                   0
4          896                   1
5          897                   0
6          898                   1
7          899                   0
8          900                   1
9          901                   0


# **Submission**

In [28]:
# Create a DataFrame with PassengerId and Survived predictions
submission = pd.DataFrame({'PassengerId': test_data['PassengerId'], 'Survived': test_predictions})

# Save the DataFrame as a CSV file for submission
submission.to_csv('/kaggle/working/submission.csv', index=False)
